# REAL-Colon detector inference

Runs the polyp detector over the 60 REAL-Colon recordings and writes per-frame detections and annotations. Each recording is downloaded from Figshare to the Colab disk, passed through the detector and deleted, so the 1 TB of frames is never stored. Recordings already processed are skipped when the notebook is run again.

Use a GPU runtime. Run sections 1 to 4 once, then section 5, then sections 6 and 7.

## 1. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
PROJECT  = pathlib.Path('/content/drive/MyDrive/realcolon_project')
WEIGHTS  = PROJECT / 'best.pt'
OUT      = PROJECT / 'realcolon'
DET_DIR  = OUT / 'detections'
ANN_DIR  = OUT / 'annotations'
META_DIR = OUT / 'meta'
for d in (DET_DIR, ANN_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

LOCAL = pathlib.Path('/content/tmp')
LOCAL.mkdir(exist_ok=True)

assert WEIGHTS.exists(), f'weights not found at {WEIGHTS}'
print('weights :', WEIGHTS)
print('output  :', OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 2. Install

In [ ]:
!pip -q install "ultralytics==8.4.144" || pip -q install "ultralytics>=8.4,<8.5"
import ultralytics, torch, cv2, numpy as np
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. List the Figshare files and download the metadata and annotation archives

In [ ]:
import json, time, hashlib, requests

ARTICLE = 22202866
def list_files(article=ARTICLE):
    files, page = [], 1
    while True:
        r = requests.get(f'https://api.figshare.com/v2/articles/{article}/files',
                         params={'page': page, 'page_size': 1000}, timeout=60)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        files += batch
        page += 1
    return files

FILES = list_files()
json.dump(FILES, open(META_DIR / 'figshare_files.json', 'w'), indent=1)
print(len(FILES), 'files listed,', round(sum(f['size'] for f in FILES) / 1e12, 2), 'TB in total')
for f in FILES[:6]:
    print(f"  {f['name']:40s} {f['size']/1e9:8.2f} GB")

def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()

def download(f, dest, chunk=1 << 24, tries=6):
    dest = pathlib.Path(dest)
    for attempt in range(1, tries + 1):
        try:
            pos = dest.stat().st_size if dest.exists() else 0
            if pos < f['size']:
                headers = {'Range': f'bytes={pos}-'} if pos else {}
                with requests.get(f['download_url'], headers=headers, stream=True, timeout=120) as r:
                    r.raise_for_status()
                    mode = 'ab' if (pos and r.status_code == 206) else 'wb'
                    with open(dest, mode) as fh:
                        for b in r.iter_content(chunk):
                            fh.write(b)
            size = dest.stat().st_size
            if size < f['size']:
                print(f"  {f['name']}: {size}/{f['size']} bytes, resuming (attempt {attempt})")
                continue
            if size == f['size'] and (not f.get('computed_md5') or _md5(dest) == f['computed_md5']):
                return dest
            print(f"  {f['name']}: size or MD5 wrong, deleting and retrying (attempt {attempt})")
            dest.unlink()
        except (requests.RequestException, OSError) as e:
            print(f"  {f['name']}: {type(e).__name__}: {e} (attempt {attempt})")
        time.sleep(min(60, 5 * attempt))
    if dest.exists():
        dest.unlink()
    raise IOError(f"{f['name']} failed after {tries} attempts")

small = [f for f in FILES if f['name'].endswith(('.csv', '.md')) or '_annotation' in f['name']]
for f in small:
    target = META_DIR / f['name'] if not '_annotation' in f['name'] else LOCAL / f['name']
    if '_annotation' in f['name'] and (ANN_DIR / f"{f['name'].split('_annotation')[0]}.json.gz").exists():
        continue
    download(f, target)
print('small files done')

## 4. Convert the annotations

In [ ]:
import pandas as pd, zipfile, tarfile
vi = pd.read_csv(META_DIR / 'video_info.csv')
li = pd.read_csv(META_DIR / 'lesion_info.csv')
print('video_info columns :', list(vi.columns))
print(vi.head(3).to_string())
print('\nlesion_info columns:', list(li.columns))
print(li.head(3).to_string())

def open_archive(path):
    path = str(path)
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            for n in z.namelist():
                if not n.endswith('/'):
                    yield n, z.read(n)
    else:
        with tarfile.open(path, 'r|*') as t:
            for m in t:
                if m.isfile():
                    yield m.name, t.extractfile(m).read()

ann_archives = sorted(LOCAL.glob('*_annotation*'))
print('\n', len(ann_archives), 'annotation archives on local disk')
if ann_archives:
    for name, data in open_archive(ann_archives[0]):
        if name.endswith('.xml'):
            print('\nexample file:', name)
            print(data.decode('utf-8', 'replace')[:1500])
            break

In [ ]:
import gzip, re, xml.etree.ElementTree as ET

def frame_index(name):
    m = re.search(r'_(\d+)(?:\.0+)?\.(?:jpg|jpeg|png|xml)$', name, re.I)
    return int(m.group(1)) if m else None

def parse_xml(data):
    root = ET.fromstring(data)
    out = []
    for obj in root.iter('object'):
        bb = obj.find('bndbox')
        if bb is None:
            continue
        g = lambda k: float(bb.find(k).text)
        uid = obj.findtext('unique_id') or obj.findtext('name') or ''
        out.append({'box': [g('xmin'), g('ymin'), g('xmax'), g('ymax')], 'lesion_id': uid.strip()})
    return out

for arc in ann_archives:
    vid = arc.name.split('_annotation')[0]
    dest = ANN_DIR / f'{vid}.json.gz'
    if dest.exists():
        continue
    ann, n_xml = {}, 0
    for name, data in open_archive(arc):
        if not name.lower().endswith('.xml'):
            continue
        n_xml += 1
        boxes = parse_xml(data)
        if boxes:
            ann[frame_index(name)] = boxes
    tmp = dest.with_name(dest.name + '.part')
    with gzip.open(tmp, 'wt') as fh:
        json.dump({'video': vid, 'n_xml': n_xml, 'annotations': ann}, fh)
    n_les = len({b['lesion_id'] for v in ann.values() for b in v})
    print(f'{vid}: {n_xml} xml files, {len(ann)} annotated frames, {n_les} lesions')
    arc.unlink()
print('annotations done:', len(list(ANN_DIR.glob('*.json.gz'))), 'recordings')

## 5. Detector inference

Same checkpoint and settings as for LDPolypVideo (`imgsz=320`, `conf=0.01`). Frames are decoded at half resolution and the boxes are scaled back to the native pixel grid. Detections below confidence 0.05 are not stored. Set `TEST_ONE = False` after checking one recording in section 6.

In [ ]:
import itertools
from concurrent.futures import ThreadPoolExecutor
from ultralytics import YOLO

TEST_ONE   = True
IMGSZ      = 320
CONF_INFER = 0.01
CONF_KEEP  = 0.05
BATCH      = 64
DECODE     = cv2.IMREAD_REDUCED_COLOR_2
SCALE      = 2.0

model = YOLO(str(WEIGHTS))
frame_files = sorted([f for f in FILES if f['name'].endswith(('_frames.zip', '_frames.tar', '_frames.tar.gz', '_frames'))
                      or '_frames' in f['name']], key=lambda f: f['name'])
print(len(frame_files), 'frame archives listed')

def decode(item):
    name, data = item
    img = cv2.imdecode(np.frombuffer(data, np.uint8), DECODE)
    return frame_index(name), img

def run_batch(batch, dets):
    idx = [i for i, _ in batch]
    imgs = [im for _, im in batch]
    res = model.predict(imgs, imgsz=IMGSZ, conf=CONF_INFER, verbose=False, device=0)
    for i, r in zip(idx, res):
        b = r.boxes
        keep = b.conf.cpu().numpy() >= CONF_KEEP
        xyxy = (b.xyxy.cpu().numpy()[keep] * SCALE).round(1).tolist()
        conf = b.conf.cpu().numpy()[keep].round(4).tolist()
        dets[i] = [{'box': bx, 'conf': c} for bx, c in zip(xyxy, conf)]

def infer(f, arc, t_wait):
    vid = f['name'].split('_frames')[0]
    dest = DET_DIR / f'{vid}.json.gz'
    t1 = time.time()
    dets, n = {}, 0
    stream = ((n_, d_) for n_, d_ in open_archive(arc) if frame_index(n_) is not None)
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
        while True:
            chunk = list(itertools.islice(stream, BATCH))
            if not chunk:
                break
            batch = list(pool.map(decode, chunk))
            n += len(batch)
            run_batch(batch, dets)
    t2 = time.time()
    tmp = dest.with_name(dest.name + '.part')
    with gzip.open(tmp, 'wt') as fh:
        json.dump({'video': vid, 'n_frames': n, 'imgsz': IMGSZ, 'conf_keep': CONF_KEEP,
                   'decode': 'IMREAD_REDUCED_COLOR_2', 'ultralytics': ultralytics.__version__,
                   'detections': dets}, fh)
    tmp.rename(dest)
    arc.unlink()
    print(f'{vid}: {n} frames | waited {t_wait/60:.1f} min for download, inference {(t2-t1)/60:.1f} min '
          f'({n/(t2-t1):.0f} frames/s) | {dest.stat().st_size/1e6:.1f} MB', flush=True)

todo = [f for f in frame_files if not (DET_DIR / f"{f['name'].split('_frames')[0]}.json.gz").exists()]
if TEST_ONE:
    todo = todo[:1]
print(len(todo), 'recordings to process')
dl = ThreadPoolExecutor(max_workers=1)
fut = dl.submit(download, todo[0], LOCAL / todo[0]['name']) if todo else None
failed = []
for j, f in enumerate(todo):
    t0 = time.time()
    try:
        arc = fut.result()
    except Exception as e:
        arc = None
        print(f"SKIPPED {f['name']}: {e}")
        failed.append(f['name'])
    t_wait = time.time() - t0
    if j + 1 < len(todo):
        fut = dl.submit(download, todo[j + 1], LOCAL / todo[j + 1]['name'])
    if arc is not None:
        try:
            infer(f, arc, t_wait)
        except Exception as e:
            print(f"FAILED inference {f['name']}: {type(e).__name__}: {e}")
            failed.append(f['name'])
            if pathlib.Path(arc).exists():
                pathlib.Path(arc).unlink()
if failed:
    print('Not finished (rerun this cell to retry them):', failed)
print('done:', len(list(DET_DIR.glob('*.json.gz'))), 'of', len(frame_files), 'recordings')

## 6. Check the output

Confirms that detections and annotations share frame indices and the pixel grid.

In [ ]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0.0

for p in sorted(DET_DIR.glob('*.json.gz')):
    D = json.load(gzip.open(p, 'rt'))
    A = json.load(gzip.open(ANN_DIR / p.name, 'rt'))
    det = {int(k): v for k, v in D['detections'].items()}
    ann = {int(k): v for k, v in A['annotations'].items()}
    hits = sum(any(iou(d['box'], g['box']) >= 0.5 for d in det.get(f, []) if d['conf'] >= 0.25) for f, gs in ann.items() for g in gs)
    nbox = sum(len(v) for v in ann.values())
    xs = [b['box'][2] for v in ann.values() for b in v]
    print(f"{D['video']}: frames {D['n_frames']} (xml {A['n_xml']}), annotated frames {len(ann)}, "
          f"lesions {len({g['lesion_id'] for v in ann.values() for g in v})}, "
          f"GT boxes hit at conf>=0.25: {hits}/{nbox} ({100*hits/max(nbox,1):.1f}%), "
          f"max GT x2 {max(xs) if xs else 'n/a'}, frame indices {min(det)}..{max(det)}")

## 7. Write the bundles used by the evaluation (copy them to data/realcolon)

In [ ]:
import gzip, json, numpy as np
BUNDLE = OUT / 'bundle'
BUNDLE.mkdir(exist_ok=True)
for centre in ['001', '002', '003', '004']:
    paths = sorted(DET_DIR.glob(f'{centre}-*.json.gz'))
    for part, chunk in (('a', paths[:8]), ('b', paths[8:])):
        V, F, C, B, AV, AF, AL, AB, meta = [], [], [], [], [], [], [], [], {}
        for p in chunk:
            D = json.load(gzip.open(p, 'rt'))
            A = json.load(gzip.open(ANN_DIR / p.name, 'rt'))
            vid = D['video']
            k = int(vid.split('-')[1])
            meta[vid] = {'n_frames': D['n_frames'], 'n_xml': A.get('n_xml')}
            for f, ds in D['detections'].items():
                for d in ds:
                    V.append(k); F.append(int(f)); C.append(d['conf']); B.append(d['box'])
            for f, gs in A['annotations'].items():
                for g in gs:
                    AV.append(k); AF.append(int(f)); AL.append(g['lesion_id']); AB.append(g['box'])
        dest = BUNDLE / f'centre_{centre}{part}.npz'
        np.savez_compressed(dest,
            vid=np.array(V, np.int16), frame=np.array(F, np.int32),
            conf=np.round(np.array(C) * 10000).astype(np.uint16),
            box=np.round(np.array(B, np.float64).reshape(-1, 4) * 10).astype(np.int32),
            a_vid=np.array(AV, np.int16), a_frame=np.array(AF, np.int32),
            a_lesion=np.array(AL), a_box=np.round(np.array(AB, np.float64).reshape(-1, 4) * 10).astype(np.int32),
            meta=np.array(json.dumps(meta)))
        print(f'{dest.name}: {len(chunk)} recordings, {len(V):,} detections, {len(AV):,} annotated boxes, '
              f'{dest.stat().st_size/1e6:.1f} MB')